In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import numpy as np
from nn import Sequential, Linear, ConvLayer, Flatten, MaxPooling2D
from activations import ReLU, Sigmoid, Tanh
from loss_functions import MSELoss
import pandas as pd
import torch
from torch.nn.functional import max_pool2d

In [2]:
x = np.random.randint(1, 50, size=(2, 2, 4, 4))

In [3]:
x

array([[[[32, 21, 45, 47],
         [41, 46, 16, 21],
         [ 7, 24, 15, 12],
         [10, 28, 36,  3]],

        [[25, 23, 35, 11],
         [11, 31, 14, 30],
         [47, 29,  5, 11],
         [19,  7,  5, 35]]],


       [[[ 5, 35, 10, 43],
         [37, 26, 37,  9],
         [39, 32, 14, 22],
         [26, 44,  4, 15]],

        [[21, 47, 44, 27],
         [35,  9, 36, 20],
         [23, 11, 36, 46],
         [ 1, 25, 31, 18]]]])

In [17]:
a = np.random.randint(0, 10, size=(5, 5))
print(a)
print(a.max(axis=(0, 1)))

[[1 6 5 7 3]
 [7 3 6 2 4]
 [4 1 2 4 0]
 [2 0 5 0 4]
 [5 8 5 2 9]]
9


In [20]:
x = np.random.randint(1, 50, size=(2, 2, 4, 4))
# Assuming x shape is [B, C, H, W]
b, c, h, w = x.shape
pool_size = 2

# 1. Reshape to isolate the 2x2 blocks:
# [B, C, H_out, pool_size, W_out, pool_size]
x_reshaped = x.reshape(b, c, h // pool_size, pool_size, w // pool_size, pool_size)

# 2. Grab the maximum over the pool axes (axes 3 and 5)
out = x_reshaped.max(axis=(3, 5))

In [21]:
x_reshaped.shape

(2, 2, 2, 2, 2, 2)

In [5]:
out

array([[[[46, 47],
         [28, 36]],

        [[31, 35],
         [47, 35]]],


       [[[37, 43],
         [44, 22]],

        [[47, 44],
         [25, 46]]]])

In [6]:
pool = MaxPooling2D(2, 2)
out = pool(x)

In [7]:
out

array([[[[46., 47.],
         [28., 36.]],

        [[31., 35.],
         [47., 35.]]],


       [[[37., 43.],
         [44., 22.]],

        [[47., 44.],
         [25., 46.]]]])

In [7]:
np.ones_like(out).astype(int)

array([[[[1, 1],
         [1, 1]],

        [[1, 1],
         [1, 1]]],


       [[[1, 1],
         [1, 1]],

        [[1, 1],
         [1, 1]]]])

In [8]:
pool.backward(out_grad=np.ones_like(out))

grad_pixel: [[[[1.]]

  [[1.]]]


 [[[1.]]

  [[1.]]]]
grad_pixel: [[[[1.]]

  [[1.]]]


 [[[1.]]

  [[1.]]]]
grad_pixel: [[[[1.]]

  [[1.]]]


 [[[1.]]

  [[1.]]]]
grad_pixel: [[[[1.]]

  [[1.]]]


 [[[1.]]

  [[1.]]]]


array([[[[0. , 1. , 0. , 0. ],
         [0. , 0. , 1. , 0. ],
         [0. , 1. , 1. , 0. ],
         [0. , 0. , 0. , 0. ]],

        [[0. , 0. , 0. , 0. ],
         [0. , 1. , 1. , 0. ],
         [0. , 1. , 0. , 0. ],
         [0. , 0. , 1. , 0. ]]],


       [[[0. , 0. , 0. , 1. ],
         [0.5, 0.5, 0. , 0. ],
         [0. , 0. , 1. , 0. ],
         [0. , 1. , 0. , 0. ]],

        [[0. , 0. , 0. , 0. ],
         [0. , 1. , 0. , 1. ],
         [0. , 0. , 1. , 0. ],
         [0.5, 0.5, 0. , 0. ]]]])

In [9]:
MaxPooling2D(2, 2)(x)

array([[[[46., 39.],
         [40., 45.]],

        [[47., 41.],
         [26., 35.]]],


       [[[47., 43.],
         [49., 15.]],

        [[46., 49.],
         [35., 44.]]]])

In [45]:
x_slice = x[:, :, 0:2, 0:2]
x_slice

array([[[[ 7, 13],
         [ 2, 10]],

        [[ 6,  6],
         [ 6, 18]],

        [[14,  6],
         [ 4, 15]]],


       [[[13,  3],
         [12, 15]],

        [[14, 19],
         [12, 17]],

        [[ 5, 19],
         [ 5, 14]]]])

In [46]:
max_val = np.max(x_slice, axis=(2, 3), keepdims=True)
max_val

array([[[[13]],

        [[18]],

        [[15]]],


       [[[15]],

        [[19]],

        [[19]]]])

In [47]:
mask = x_slice == max_val
mask

array([[[[False,  True],
         [False, False]],

        [[False, False],
         [False,  True]],

        [[False, False],
         [False,  True]]],


       [[[False, False],
         [False,  True]],

        [[False,  True],
         [False, False]],

        [[False,  True],
         [False, False]]]])

In [48]:
out_grad = np.random.randint(1, 20, (2, 3, 2, 2))
out_grad

array([[[[12,  1],
         [14,  4]],

        [[12, 16],
         [ 3, 11]],

        [[11,  8],
         [16, 12]]],


       [[[ 1,  6],
         [ 9, 11]],

        [[ 7, 17],
         [ 2, 17]],

        [[ 6, 14],
         [19, 16]]]])

In [31]:
grad_pixel = out_grad[:, :, 0, 0, None, None]
grad_pixel

array([[[[12]],

        [[ 4]],

        [[ 3]]],


       [[[ 7]],

        [[ 5]],

        [[17]]]])

In [33]:
(mask * grad_pixel) / np.sum(mask, axis=(2, 3), keepdims=True)

array([[[[ 0.  , 12.  ],
         [ 0.  ,  0.  ]],

        [[ 0.  ,  0.  ],
         [ 4.  ,  0.  ]],

        [[ 0.  ,  1.5 ],
         [ 1.5 ,  0.  ]]],


       [[[ 1.75,  1.75],
         [ 1.75,  1.75]],

        [[ 0.  ,  0.  ],
         [ 5.  ,  0.  ]],

        [[ 8.5 ,  8.5 ],
         [ 0.  ,  0.  ]]]])